# 第61课 · nn.Module 实战 — Linear / ReLU / Sequential，参数管理与模型保存

**目标**：用 `nn.Module` 复现 `L57` 中手写的 MLP，使两者在同一数据上参数数量和前向输出形状完全一致。

🔗 **Aurora 连接**：Month 2 关键词识别模型（KWS）、Month 3 ASR 微调、Month 5 LLM 都以 `nn.Module` 为骨架——掌握它的注册机制和序列化（serialization）接口是后续一切工作的前提。

← **上一课**　[L60 · autograd 机制](L60_pytorch_autograd.ipynb)

> 上节课学习了 **autograd 机制**：gradfn 计算图，backward()，retaingraph 原理。  
> 本课将探讨 **nn.Module 实战**。

## 本课剧情：参数"住"在哪里，决定了模型能不能被保存和加载

L57 的 `MLP` 类能前向、能反向——但有个隐患：**参数是 `Value` 对象，散落在 Python 变量里**。想保存模型到磁盘？得手工收集每个权重。想把模型搬到 GPU？要一个个转。

`nn.Module` 解决了这个问题，像一个"参数户籍系统"：

```python
class MyLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(4, 4))  # ← 登记为参数
    
    def forward(self, x):
        return x @ self.weight
```

只要用 `nn.Parameter` 包裹，`self.weight` 就自动出现在 `model.parameters()`，被优化器发现，被 `state_dict()` 收录，用 `model.to('cuda')` 一键搬到 GPU。

**三个关键工具**：

| 工具 | 作用 |
|---|---|
| `nn.Parameter(tensor)` | 注册为可训练参数 |
| `model.parameters()` | 迭代器，返回所有参数 |
| `model.state_dict()` | 字典，保存/加载模型状态 |

**Aurora 连接**：`aurora.llm.KVCache` 继承 `nn.Module`，键值缓存用 `register_buffer()` 登记，不参与梯度但跟着模型走设备。

本节任务：实现 `AudioMLP(nn.Module)` — 两层线性 + ReLU，参数总量 = 40×64+64 + 64×10+10 = 3,274。

In [ ]:
import torch
import torch.nn as nn

## 1. `forward()` 与参数自动注册

`nn.Module` 要求子类实现 `forward()`，调用时写 `model(x)` 而非 `model.forward(x)`——
前者会触发钩子（hooks），后者绕过它们。

参数注册有两种方式：
- 赋值内置层（`nn.Linear`, `nn.Conv1d` …）：参数自动注册
- 赋值裸张量：**必须** 用 `nn.Parameter(tensor)` 包装，否则不会被追踪

```python
# 正确：参数被注册
self.w = nn.Parameter(torch.randn(10, 5))
# 错误：不被追踪，优化器看不到它
self.w = torch.randn(10, 5)
```

In [ ]:
# 演示：nn.Parameter 注册
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)           # 自动注册 weight + bias
        self.scale = nn.Parameter(torch.ones(1))  # 手动注册
        self.buf = torch.zeros(1)            # 未注册，仅普通张量

    def forward(self, x):
        return self.fc(x) * self.scale

tiny = TinyNet()
param_names = [n for n, _ in tiny.named_parameters()]
print('注册的参数:', param_names)  # fc.weight, fc.bias, scale
print('buf 未出现:', 'buf' not in param_names)

### `register_buffer()` — 正确的非参数状态注册方式

`buf` 的反例展示了**不注册**的后果；正确做法是用 `register_buffer()`，适用于不需要梯度但需要随模型移动（`.to(device)`）并出现在 `state_dict()` 中的张量（如 BatchNorm 的 running mean）。

> 区别一览：
>
> | 方式 | `parameters()` | `state_dict()` | 梯度 |
> |---|---|---|---|
> | `nn.Parameter` | ✅ | ✅ | ✅ |
> | `register_buffer()` | ❌ | ✅ | ❌ |
> | 裸张量属性 | ❌ | ❌ | ❌ |


In [ ]:
# 演示：register_buffer — 出现在 state_dict 但不在 parameters()
class TinyNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.scale = nn.Parameter(torch.ones(1))   # 有梯度，出现在 parameters()
        self.register_buffer('running_mean', torch.zeros(1))  # 无梯度，出现在 state_dict
        self.plain_buf = torch.zeros(1)             # 不注册，state_dict 中不出现

    def forward(self, x):
        return self.fc(x) * self.scale

tiny2 = TinyNetV2()
print('state_dict keys:', list(tiny2.state_dict().keys()))
# 预期: fc.weight, fc.bias, scale, running_mean  (不含 plain_buf)
assert 'running_mean' in tiny2.state_dict(), 'register_buffer 的张量应出现在 state_dict'
assert 'plain_buf' not in tiny2.state_dict(), '裸张量不出现在 state_dict'
assert 'running_mean' not in [n for n, _ in tiny2.named_parameters()], 'buffer 不计入梯度参数'
print('✅ register_buffer 验证通过：running_mean 在 state_dict 中，不在 parameters() 中')


## 2. `named_parameters()` 与 `state_dict()`

`named_parameters()` 递归遍历所有子模块，返回 `(名称, 张量)` 对；常用于统计参数量或做分层学习率。

```python
total = sum(p.numel() for p in model.parameters())
```

`state_dict()` 返回 `OrderedDict`，键为参数名，值为张量——这是保存/加载检查点（checkpoint，ckpt）的标准接口：

```python
torch.save(model.state_dict(), 'ckpt.pt')
model.load_state_dict(torch.load('ckpt.pt'))
```

两个模型结构相同时，可直接把 `state_dict` 从一个传给另一个，实现权重复制。

In [ ]:
# 演示：state_dict 键名与形状
for name, param in tiny.named_parameters():
    print(f'{name:20s}  shape={list(param.shape)}  numel={param.numel()}')

print('\nstate_dict keys:', list(tiny.state_dict().keys()))

## 3. `nn.Sequential`：省掉 `forward()`

`nn.Sequential` 按顺序串联子模块，自动实现 `forward(x) = layer_n(... layer_1(x))`，无需手写循环。子模块以整数索引命名（`0`, `1`, `2` …），也可用 `OrderedDict` 给定字符串名。

```python
net = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
# 等价于：x -> Linear -> ReLU -> Linear -> 输出
```

适用条件：层之间**无分支、无跳连（skip connection）**。有残差连接（residual connection）时必须回到手写 `forward()`。

In [ ]:
# 演示：nn.Sequential 前向
seq = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
x_demo = torch.randn(8, 40)
print('Sequential 输出形状:', seq(x_demo).shape)  # (8, 10)
print('子模块列表:', list(seq.children()))

## 4. ✏️ 实现 `class AudioMLP(nn.Module)`

**架构**：`Linear(in→hidden) → ReLU → Linear(hidden→out)`

**实现模板**：
```python
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )
    
    def forward(self, x):
        return self.layers(x)
```

**为什么用 `nn.Sequential`**：省去手写 `forward()` 里的链式调用，模块顺序即计算顺序。

**验收标准**：

| 检查项 | 期望值 |
|---|---|
| `m(torch.randn(8,40)).shape` | `(8, 10)` |
| `sum(p.numel() for p in m.parameters())` | `3274` |
| `m.layers[0].weight.shape` | `(64, 40)` |

In [ ]:
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        # ✏️ TODO: 定义 self.layers = nn.Sequential(...)
        raise NotImplementedError("TODO: 在 __init__ 中定义 self.layers = nn.Sequential(nn.Linear(in_features, hidden), nn.ReLU(), nn.Linear(hidden, out_features))")

    def forward(self, x):
        # ✏️ TODO: return self.layers(x)
        raise NotImplementedError("TODO: 在 forward 中返回 self.layers(x)")


In [ ]:
try:
    m = AudioMLP(40, 64, 10)
    out = m(torch.randn(8, 40))
    assert out.shape == (8, 10), f'期望 (8,10)，得到 {out.shape}'
    print('✅ AudioMLP 前向输出形状正确:', out.shape)
except NotImplementedError as e:
    print(f'⚠️ 练习未完成，请先实现 AudioMLP：{e}')


## 5. 参数实验：验证参数总量

对 `AudioMLP(40, 64, 10)` 手算参数数量：

```
layers.0 (Linear 40→64): weight=40×64=2560, bias=64     → 2624
layers.2 (Linear 64→10): weight=64×10=640,  bias=10     →  650
合计: 2624 + 650 = 3274
```

**预期现象**：`sum(p.numel() for p in m.parameters())` 返回 `3274`，与手写 `L57` 的同构网络完全一致（若手写版有偏置的话）。

修改 `hidden=128` 后总量变为 `40×128+128 + 128×10+10 = 5248 + 1290 = 6538`，可自行验证。

In [ ]:
try:
    total = sum(p.numel() for p in m.parameters())
    print(f'参数总量: {total}')
    assert total == 3274, f'期望 3274，得到 {total}'
    print('✅ 参数总量与手算一致')

    # 查看每层参数明细
    for name, p in m.named_parameters():
        print(f'  {name:25s}  {list(p.shape)}  = {p.numel()}')

    # 实验：hidden=128 时参数数量
    m128 = AudioMLP(40, 128, 10)
    print(f'\nhidden=128 时参数总量: {sum(p.numel() for p in m128.parameters())}')
except NotImplementedError as e:
    print(f'⚠️ 练习未完成，请先实现 AudioMLP：{e}')


## 本课收束

`AudioMLP.forward()` 通过 `nn.Sequential` 把线性变换和 ReLU 串联，输出形状 `(batch, out_features)`，与手写 `L57` 的数学等价但参数生命周期由 `nn.Module` 统一管理。
`named_parameters()` 和 `state_dict()` 是 Aurora Month 2 KWS 模型保存检查点、Month 3 ASR 加载预训练权重的直接基础。
下一节（L62）将把特征提取器包装进 `torch.utils.data.Dataset`，构建自定义 `__getitem__`，完成音频数据批量加载。

## ✏️ 白板挑战：nn.Module 参数管理手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`AudioMLP(40, 64, 10)` 共有多少个参数？逐层列出。  
（L1: 40×64+64=2624，L2: 64×10+10=650，共2624+650=3274）

**问 2**：`nn.Parameter` 和普通 `torch.Tensor` 赋值给 `self.x` 有什么区别？  
（Parameter 自动出现在 parameters()；普通 Tensor 不会，不被优化器更新）

**问 3**：`model.state_dict()` 和 `model.parameters()` 有什么区别？  
（state_dict包含所有buffer和parameter，字典格式；parameters()只返回Parameter，迭代器格式）

**问 4**：`model.to('cuda')` 为什么只对注册的 `Parameter` 和 `buffer` 有效？  
（Module只知道通过register_parameter/register_buffer登记的张量；散落变量不在Module管辖范围）

**问 5**：`nn.Sequential([Linear, ReLU, Linear])` 的 `forward(x)` 等价于什么代码？  
（等价于 x1=Linear1(x); x2=ReLU(x1); out=Linear2(x2); return out — 顺序调用每个子模块）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch, torch.nn as nn

# 问1：参数总量
try:
    m_q = AudioMLP(40, 64, 10)
    total_q = sum(p.numel() for p in m_q.parameters())
    assert total_q == 3274, f"参数总量={total_q}"
    l1 = m_q.layers[0].weight.numel() + m_q.layers[0].bias.numel()
    l2 = m_q.layers[2].weight.numel() + m_q.layers[2].bias.numel()
    print(f"Q1 ✅  AudioMLP(40,64,10)参数: L1={l1}(40×64+64), L2={l2}(64×10+10), 共{total_q}")
except NotImplementedError:
    print(f"Q1 ✅  参数总量=40×64+64+64×10+10=2624+650=3274（待实现后验证）")

# 问2：Parameter vs 普通 Tensor
class _TestMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = nn.Parameter(torch.zeros(3))
        self.t = torch.zeros(3)  # 不注册
    def forward(self, x): return x
tm = _TestMod()
param_names = [n for n, _ in tm.named_parameters()]
assert 'p' in param_names and 't' not in param_names
print(f"Q2 ✅  Parameter注册在parameters()中，普通Tensor不在: {param_names}")

# 问3：state_dict vs parameters
class _TestMod2(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.ones(2))
        self.register_buffer('buf', torch.zeros(2))
    def forward(self, x): return x
tm2 = _TestMod2()
sd_keys = list(tm2.state_dict().keys())
param_keys = [n for n, _ in tm2.named_parameters()]
assert 'buf' in sd_keys and 'buf' not in param_keys
print(f"Q3 ✅  state_dict={sd_keys}(含buffer), parameters={param_keys}(仅Parameter)")

# 问4：to()设备迁移（只测CPU→CPU，不依赖CUDA）
try:
    m_q2 = AudioMLP(4, 8, 2)
    m_q2_cpu = m_q2.to('cpu')
    assert all(p.device.type == 'cpu' for p in m_q2_cpu.parameters())
    print(f"Q4 ✅  model.to('cpu')使所有Parameter迁移到CPU")
except NotImplementedError:
    print(f"Q4 ✅  Module.to(device)递归迁移所有注册的Parameter和Buffer（待实现后验证）")

# 问5：Sequential等价展开
seq_q = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
x_q = torch.randn(3, 4)
out_seq = seq_q(x_q)
out_manual = seq_q[2](seq_q[1](seq_q[0](x_q)))
assert torch.allclose(out_seq, out_manual)
print(f"Q5 ✅  Sequential(x)等价于逐层调用，输出shape={tuple(out_seq.shape)}")
print("\n🎉 nn.Module 参数管理白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l61_review = {
    "nn_parameter_understood":  None,  # 理解nn.Parameter让张量进入parameters()？True/False
    "audio_mlp_impl":           None,  # AudioMLP实现正确，shape/参数总量验证通过？True/False
    "state_dict_understood":    None,  # 理解state_dict包含buffer+parameter，用于保存/加载？True/False
    "sequential_forward":       None,  # 理解Sequential.forward = 顺序调用子模块？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l61_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l61_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L61 全部通关！进入 L62：Dataset 与 DataLoader')

In [ ]:
# Aurora 连接：KVCache 继承 nn.Module，使用相同的参数注册机制
from aurora.llm.kvcache import KVCache  # KVCache 继承 nn.Module
print("aurora.llm.kvcache.KVCache:", KVCache)
print("bases:", [b.__name__ for b in KVCache.__bases__])

---

→ **下一课**　[L62 · Dataset 与 DataLoader](L62_kws_dataset.ipynb)

> 下节课将学习 **Dataset 与 DataLoader**：自定义 getitem，音频数据批量加载。